In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "POLUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,0.2140,0.2142,0.2136,0.2137,95720.9,2025-06-01 00:04:59.999999+00:00,20471.83248,121,72999.9,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.000000,0.000000e+00,0.000000e+00,NaN,NaN
1,2025-06-01 00:05:00+00:00,0.2137,0.2138,0.2136,0.2138,24108.0,2025-06-01 00:09:59.999999+00:00,5151.14851,40,7831.3,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.000002,1.246439e-06,9.971510e-07,NaN,NaN
2,2025-06-01 00:10:00+00:00,0.2138,0.2140,0.2134,0.2136,124953.6,2025-06-01 00:14:59.999999+00:00,26696.71412,118,29028.5,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000003,-6.345646e-07,-2.708645e-06,NaN,NaN
3,2025-06-01 00:15:00+00:00,0.2136,0.2136,0.2132,0.2133,33999.9,2025-06-01 00:19:59.999999+00:00,7253.54456,73,9719.7,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000017,-6.054308e-06,-1.057934e-05,NaN,NaN
4,2025-06-01 00:20:00+00:00,0.2133,0.2137,0.2132,0.2136,59750.9,2025-06-01 00:24:59.999999+00:00,12754.94294,111,42768.2,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000012,-7.694388e-06,-3.873214e-06,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]
fwd_ret_train = train_df[ret_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]
fwd_ret_valid = valid_df[ret_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret_test = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,441
[info] optuna train rows: 53,401
[info] valid rows:        13,351
[info] test rows:         16,689


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    fwd_ret_valid=fwd_ret_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-23 13:57:31,887] A new study created in memory with name: no-name-914efa8e-1469-42cc-83d2-ff37942b679d


[I 2026-03-23 13:57:32,194] Trial 0 finished with value: 0.5418571193144661 and parameters: {'n_estimators': 400, 'learning_rate': 0.09423875899553878, 'max_depth': 5, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 3, 'reg_lambda': 0.13066739238053282, 'scale_pos_weight': 1.3936108028730012}. Best is trial 0 with value: 0.5418571193144661.


[I 2026-03-23 13:57:32,319] Trial 1 finished with value: 0.5263066075990449 and parameters: {'n_estimators': 600, 'learning_rate': 0.07036510754376854, 'max_depth': 3, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_lambda': 0.23102018878452935, 'scale_pos_weight': 0.9574879085555049}. Best is trial 0 with value: 0.5418571193144661.


[I 2026-03-23 13:57:32,562] Trial 2 finished with value: 0.5466785089811701 and parameters: {'n_estimators': 400, 'learning_rate': 0.05642937488667569, 'max_depth': 4, 'subsample': 0.7873687420594125, 'colsample_bytree': 0.8447411578889518, 'min_child_weight': 3, 'reg_lambda': 0.3839629299804172, 'scale_pos_weight': 1.05879874492964}. Best is trial 2 with value: 0.5466785089811701.


[I 2026-03-23 13:57:32,728] Trial 3 finished with value: 0.5344890544216693 and parameters: {'n_estimators': 500, 'learning_rate': 0.0772099153368949, 'max_depth': 3, 'subsample': 0.8542703315240835, 'colsample_bytree': 0.836965827544817, 'min_child_weight': 2, 'reg_lambda': 1.6409286730647923, 'scale_pos_weight': 0.9507321330175988}. Best is trial 2 with value: 0.5466785089811701.


...

[I 2026-03-23 13:58:04,968] Trial 96 finished with value: 0.5420797803682286 and parameters: {'n_estimators': 800, 'learning_rate': 0.06912548501892063, 'max_depth': 5, 'subsample': 0.8125161581238725, 'colsample_bytree': 0.9446316659172671, 'min_child_weight': 9, 'reg_lambda': 6.121747687879499, 'scale_pos_weight': 1.482925228060626}. Best is trial 82 with value: 0.5543342496540219.


[I 2026-03-23 13:58:05,212] Trial 97 finished with value: 0.5387009775651699 and parameters: {'n_estimators': 700, 'learning_rate': 0.0645544998781675, 'max_depth': 6, 'subsample': 0.7733101636629398, 'colsample_bytree': 0.9534748702009496, 'min_child_weight': 10, 'reg_lambda': 3.898201633329687, 'scale_pos_weight': 1.3239811822622312}. Best is trial 82 with value: 0.5543342496540219.


[I 2026-03-23 13:58:05,505] Trial 98 finished with value: 0.5420403518642564 and parameters: {'n_estimators': 800, 'learning_rate': 0.06674564285939291, 'max_depth': 5, 'subsample': 0.8229188983297748, 'colsample_bytree': 0.9377053169250621, 'min_child_weight': 8, 'reg_lambda': 5.538607975682468, 'scale_pos_weight': 1.2716566378746164}. Best is trial 82 with value: 0.5543342496540219.


[I 2026-03-23 13:58:05,771] Trial 99 finished with value: 0.5428965652369249 and parameters: {'n_estimators': 700, 'learning_rate': 0.0706488416929395, 'max_depth': 5, 'subsample': 0.7418062263270285, 'colsample_bytree': 0.9250140516217966, 'min_child_weight': 9, 'reg_lambda': 4.6383612550683715, 'scale_pos_weight': 1.3539832842388573}. Best is trial 82 with value: 0.5543342496540219.


['hour_sin', 'hour_cos', 'dom_sin', 'vol_30', 'is_trending', 'month_cos', 'dow_sin', 'mom_60', 'dom_cos', 'range_15', 'dist_ma_15', 'macd_hist', 'dow_cos', 'dist_ma_30', 'vol_regime_ratio', 'vol_15', 'imbalance_15', 'atr_norm', 'month_sin', 'dist_ma_15_z', 'trend_strength', 'mom_30', 'vol_5', 'range_5', 'mom_5']
feature
hour_sin            11.672037
hour_cos            11.309538
dom_sin             11.308775
vol_30              11.220552
is_trending         11.151642
month_cos           11.091517
dow_sin             10.827024
mom_60              10.805985
dom_cos             10.632154
range_15            10.603293
dist_ma_15          10.590893
macd_hist           10.504403
dow_cos             10.444728
dist_ma_30          10.376813
vol_regime_ratio    10.317378
vol_15              10.145065
imbalance_15        10.132516
atr_norm            10.056149
month_sin           10.044533
dist_ma_15_z        10.019376
trend_strength       9.825121
mom_30               9.787908
vol_5             

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

In [11]:
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

train_pred = base_model.predict_proba(X_train_full_sel)[:, 1]
test_pred = base_model.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_ic = spearmanr(train_pred, fwd_ret_train)[0]
test_ic = spearmanr(test_pred, fwd_ret_test)[0]

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train IC:        {train_ic:.6f}")
print(f"Test IC:         {test_ic:.6f}")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:        0.364997
Test IC:         0.010007
Train ROC AUC:   0.710193
Test ROC AUC:    0.502705
Train PR AUC:    0.693267
Test PR AUC:     0.446729
Train Log Loss:  0.657433
Test Log Loss:   0.714610
Train Brier:     0.232693
Test Brier:      0.260373
Train Accuracy:  0.601225
Test Accuracy:   0.478279
Train Precision: 0.552303
Test Precision:  0.449387
Train Recall:    0.864664
Test Recall:     0.764405
Train F1:        0.674054
Test F1:         0.566017


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret_test": fwd_ret_test.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.247, 0.455] -0.000251   1669  0.007237
(0.455, 0.489] -0.000146   1669  0.006826
(0.489, 0.512] -0.000013   1669  0.007758
(0.512, 0.529] -0.000100   1669  0.006792
(0.529, 0.543]  0.000344   1669  0.006294
(0.543, 0.559] -0.000162   1668  0.006441
(0.559, 0.575]  0.000019   1669  0.006549
(0.575, 0.594] -0.000180   1669  0.006473
(0.594, 0.621]  0.000049   1669  0.006093
(0.621, 0.876] -0.000405   1669  0.009382


/tmp/ipykernel_1304534/3344132490.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret_test"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret_test"].mean())
overall_mean_ret = float(eval_df["fwd_ret_test"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret_test"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/POLUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": float(train_ic),
    "test_ic": float(test_ic),
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/POLUSDT__h6_model.joblib
[saved] features -> models/xgb/POLUSDT__h6_feature_cols.json
[saved] feature importance -> models/xgb/POLUSDT__h6_feature_importance.csv
[saved] metadata -> models/xgb/POLUSDT__h6_meta.json
